In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/data-selected-columns.csv')

print("Shape:", df.shape)
df.head()

Shape: (4600, 10)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4


In [3]:
print("\nColumns:", df.columns.tolist())


Columns: ['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition']


In [4]:
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
df.head()

date            object
price          float64
bedrooms       float64
bathrooms      float64
sqft_living      int64
sqft_lot         int64
floors         float64
waterfront       int64
view             int64
condition        int64
dtype: object

Missing values:
date           0
price          0
bedrooms       0
bathrooms      0
sqft_living    0
sqft_lot       0
floors         0
waterfront     0
view           0
condition      0
dtype: int64


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4


In [5]:
df['year_sold'] = pd.to_datetime(df['date']).dt.year
df = df.drop(columns=['date'])
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,year_sold
0,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,2014
1,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,2014
2,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,2014
3,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,2014
4,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,2014


In [6]:

median_price = df['price'].median()
df['will_sell'] = (df['price'] <= median_price).astype(int)

print("Median price:", median_price)

print(df['will_sell'].value_counts())

Median price: 460943.46153850004
will_sell
1    2300
0    2300
Name: count, dtype: int64


In [7]:
X = df[['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
        'floors', 'waterfront', 'view', 'condition', 'year_sold']]

y = df['will_sell']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4600, 9)
y shape: (4600,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples: ", len(X_test))

Training samples: 3680
Testing samples:  920


In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print()
print(classification_report(y_test, y_pred, target_names=['Wont sell', 'Will sell']))

Accuracy: 73.7 %

              precision    recall  f1-score   support

   Wont sell       0.72      0.75      0.74       450
   Will sell       0.75      0.73      0.74       470

    accuracy                           0.74       920
   macro avg       0.74      0.74      0.74       920
weighted avg       0.74      0.74      0.74       920



In [11]:
import pandas as pd

new_house = pd.DataFrame({
    'bedrooms':    [3],
    'bathrooms':   [2],
    'sqft_living': [1800],
    'sqft_lot':    [5000],
    'floors':      [2],
    'waterfront':  [0],
    'view':        [0],
    'condition':   [3],
    'year_sold':   [2023]
})

prediction  = model.predict(new_house)[0]
probability = model.predict_proba(new_house)[0][1]

if prediction == 1:
    print(f"This house WILL sell  (confidence: {round(probability*100, 1)}%)")
else:
    print(f"This house will NOT sell  (confidence: {round((1-probability)*100, 1)}%)")

This house will NOT sell  (confidence: 65.0%)


In [12]:
feature_names = X.columns.tolist()
importances   = model.feature_importances_

print("Feature importances:")
for name, score in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = '█' * int(score * 40)
    print(f"  {name:<12} {bar} {round(score*100, 1)}%")

Feature importances:
  sqft_living  ████████████████ 41.9%
  sqft_lot     ███████████ 28.7%
  bathrooms    ████ 12.2%
  bedrooms     ██ 5.6%
  floors       █ 4.8%
  condition    █ 4.1%
  view         █ 2.5%
  waterfront    0.1%
  year_sold     0.0%
